# DSPy Primitives & Tools — The Data Types and Utilities Under Everything

**Week 6 | Notebook 11 of 12**

**What you'll learn:**
- All 8 primitives from the [API reference](https://dspy.ai/current/api/): `Example`,
  `Prediction`, `Image`, `Audio`, `Code`, `History`, `Tool`, `ToolCalls` — with real payloads
- The tools family: `Embeddings` (embedding retriever), `ColBERTv2` (late-interaction
  retriever), `PythonInterpreter` (the sandbox behind PoT/CodeAct/RLM)
- How these types flow through every call you've made in the earlier notebooks

**Runtime:** ~15 minutes

**Note:** where a live demo needs an external service that isn't reachable from this
environment (the public ColBERTv2 demo server, an audio-capable model on this API key), the
cell shows the exact working code plus a note — everything else runs live, including a real
image question against a real chart and a real embedding-retrieval query.

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/11_primitives_tools.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/11_primitives_tools.ipynb
Task:      Primitives + retrieval/code tools
Calls:     ~12

With GPT-4o:       $0.18 USD
With GPT-4o-mini:  $0.02 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup

In [2]:
import dspy

from src.config import get_dspy_lm, print_config

print_config()

lm = get_dspy_lm()
dspy.configure(lm=lm)

# A real image checked into this repo (Our World in Data chart — see Notebook in 02_outlines
# for its provenance); used in the Image primitive demo below.
CHART_IMG = "../../data/invoice_samples/chart_energy_safety.png"

print(f"\n✅ DSPy configured with: {lm.model}")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      openai
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.6-flash
  groq model:      openai/gpt-oss-120b
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ DSPy configured with: openai/gpt-4o


## 2. Where Primitives Live in a DSPy Call

Every DSPy call you've ever written is assembled from these building blocks: inputs
arrive as **`Example`**s (or the multimodal types **`Image`** / **`Audio`** / **`Code`**), the
LM's reply comes back as a **`Prediction`**, agents record their steps in **`History`** and
**`ToolCalls`**, and functions become LM-callable through **`Tool`**.

```mermaid
flowchart LR
    subgraph Inputs
        E["Example"]
        I["Image"]
        A["Audio"]
        C["Code"]
    end
    Inputs --> M["dspy.Module"]
    M --> AD["Adapter"]
    AD --> LM["Language Model"]
    LM --> AD
    AD --> P["Prediction"]
    P --> H["History / ToolCalls"]
```

The Tools half of this notebook is different in kind: **retrievers and a code sandbox** —
utilities your programs call, not types the framework returns.

## 3. `Example` — The Dataset Atom

**What it is:** one labeled data point — the universal currency of trainsets, devsets,
and evaluation.

**How it works:** a dict-like record; `.with_inputs("question")` marks which fields are inputs
(vs. labels) — that distinction drives optimizers, retrieval, and evaluation. `example.inputs()`
returns only the input fields.

**When to use:** everywhere data crosses into DSPy. **Pitfall:** forgetting `.with_inputs()` —
optimizers then can't tell prompts from answers.

In [3]:
from src.datasets import generate_qa_pairs

data = generate_qa_pairs(6)
examples = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question") for d in data
]

ex = examples[0]
print(f"Full example:  {ex}")
print(f"Input fields:  {ex.inputs()}")  # only 'question' — because of with_inputs
labels = {k: ex[k] for k in ex if k not in ex.inputs()}
print(f"Label fields:  {labels}")

Full example:  Example({'question': 'What is DSPy?', 'answer': 'DSPy is a framework for programming language models.'}) (input_keys={'question'})
Input fields:  Example({'question': 'What is DSPy?'}) (input_keys={'question'})
Label fields:  {'answer': 'DSPy is a framework for programming language models.'}


## 4. `Prediction` — The Universal Return Type

**What it is:** what every module returns — a typed bag of output fields.

**How it works:** attribute access (`pred.answer`), dict-like access (`pred["answer"]`), and
the `.completions` accessor for multi-sample calls (`n > 1`). Optimizers, metrics, and your own
code all consume this one type.

**When to use:** returned to you, not constructed by you — except in metrics/tests, where
`dspy.Prediction(answer="...")` fabricates a model reply (as the evaluation notebook showed).

In [4]:
predictor = dspy.Predict("question -> answer")
pred = predictor(question="What is the capital of Iceland?")

print(f"Attribute access: {pred.answer}")
print(f"Dict-like access: {pred['answer']}")
print(f"Fields: {list(pred.keys())}")

# Multi-sample calls return several completions on the same Prediction
multi = predictor(question="Name a primary color.", config={"n": 2})
print(f"\nCompletions sampled: {len(multi.completions)}")

Attribute access: Reykjavik
Dict-like access: Reykjavik
Fields: ['answer']

Completions sampled: 2


## 5. `Image` — Multimodal Input Type

**What it is:** a pydantic type that wraps an image so it can ride along a signature
field into any vision-capable model.

**How it works:** `dspy.Image.from_path(...)` (local file) or `.from_url(...)`; the adapter
converts it to the API's image payload (base64 data URI). Works anywhere a `str` input would —
`Predict`, `ChainOfThought`, ReAct agents.

**When to use:** document QA, chart understanding, screenshot automation. **Cost note:** vision
tokens are priced like text tokens — a 1280px image is a few hundred tokens, negligible.

In [5]:
from pathlib import Path

chart = dspy.Image.from_path(Path(CHART_IMG))

qa = dspy.Predict("image, question -> answer")
pred = qa(image=chart, question="According to the chart, how many deaths per TWh does coal cause?")

print("Question: According to the chart, how many deaths per TWh does coal cause?")
print(f"Answer: {pred.answer}")
print("(a REAL image — the Our World in Data energy-safety chart from data/invoice_samples/)")

Question: According to the chart, how many deaths per TWh does coal cause?
Answer: According to the chart, coal causes 24.6 deaths per terawatt-hour (TWh).
(a REAL image — the Our World in Data energy-safety chart from data/invoice_samples/)


## 6. `Audio` — Multimodal Input Type

**What it is:** the audio counterpart of `Image` — wraps audio for audio-capable
models.

**How it works:** `dspy.Audio.from_path(...)` / `.from_url(...)`; the adapter sends it as an
inline audio payload. Pair it with an audio-capable model via `dspy.context(lm=...)` — the main
LM from `src.config` is text+vision only.

> ⚠️ **Environment note (verified):** the `gpt-4o-audio-preview` family this feature is
> documented against has been retired on the API key used for this repo, so the live call is
> shown-not-run here. The WAV generation and `Audio` payload inspection below are real; to run
> the final call, point `audio_lm` at any audio-capable model available on your key.


In [6]:
import math
import struct
import wave

# Generate a REAL audio file: a 2-second 440 Hz sine tone (concert A), 16 kHz mono WAV
wav_path = "./sample_tone.wav"
framerate = 16000
with wave.open(wav_path, "w") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(framerate)
    w.writeframes(
        b"".join(
            struct.pack("<h", int(12000 * math.sin(2 * math.pi * 440 * t / framerate)))
            for t in range(framerate * 2)
        )
    )

# Load it the way signatures expect
tone = dspy.Audio.from_path(wav_path)
print(f"Audio payload: {tone.audio_format}, {len(tone.data) // 1024} KB base64")
print(f"(a real {framerate * 2 / framerate:.0f}s WAV — a pure 440 Hz tone)")

# 🚧 LIVE CALL SHOWN FOR COMPLETENESS — needs an audio-capable model on your API key:
# audio_lm = dspy.LM("openai/<audio-capable-model>")
# with dspy.context(lm=audio_lm):
#     transcriber = dspy.Predict("audio -> description")
#     pred = transcriber(audio=tone)
# print(pred.description)

Audio payload: wav, 83 KB base64
(a real 2s WAV — a pure 440 Hz tone)


## 7. `Code` — Source Code as a First-Class Field

**What it is:** *(experimental)* a type for passing source code through signatures as
structured data (`code` + `language`) rather than an opaque string.

**How it works:** pydantic model; the adapter renders it like any other field. Enables
code-as-input tasks (review, explain, translate) and code-as-output pipelines.

**When to use:** code review/explanation agents, code-generation pipelines. **Avoid for** just
embedding a snippet in a long prompt — a plain string field is simpler until you need the
structure.

In [7]:
snippet = dspy.Code(
    code="def add(a, b):\n    return a + b",
    language="python",
)
print(f"Structured code input: language={snippet.language}, {len(snippet.code.splitlines())} lines")

reviewer = dspy.Predict("code -> review")
pred = reviewer(code=snippet.code)
print(f"\nModel review: {pred.review}")

Structured code input: language=python, 2 lines

Model review: The function `add` is correctly implemented to add two numbers. It takes two parameters `a` and `b`, and returns their sum using the `+` operator. The code is clean and follows a simple and efficient design. However, you may want to consider adding type hints to indicate that the parameters and return value should be of numeric types (e.g., `int` or `float`) if that is a requirement. Additionally, adding a docstring to describe the function's purpose could be beneficial for documentation purposes. 

Overall, here's how you could enhance it with these additions:

```python
def add(a: float, b: float) -> float:
    """Return the sum of two numbers."""
    return a + b
```

These enhancements improve code readability and maintainability.


## 8. `History` — Typed Conversation Memory

**What it is:** the message-list type behind conversational modules — most visibly
ReActV2's agent loop (Notebook 7), where each turn appends a typed event.

**How it works:** a pydantic model wrapping `messages: list[dict]`; events are plain dicts
(inputs, thoughts, tool_calls, results), so it serializes cleanly to JSON for logging and
resumes. Pass a `History` into a module to continue a prior conversation.

**When to use:** multi-turn agents, chat persistence, trace inspection. **Pitfall:** histories
grow — truncate or summarize long sessions before context limits bite.

In [8]:
history = dspy.History(
    messages=[
        {"question": "What is DSPy?", "response": "A framework for programming language models."},
        {"question": "What is RAG?", "response": "Retrieval-augmented generation."},
    ]
)

# Agents append events as they act — e.g. a tool call turn:
history.messages.append(
    {
        "question": "And which retriever should I use?",
        "tool_calls": [{"name": "retrieve", "args": {"q": "retrievers"}}],
    }
)

print(f"Events in history: {len(history.messages)}")
for event in history.messages:
    print(f"  {list(event.keys())}")

Events in history: 3
  ['question', 'response']
  ['question', 'response']
  ['question', 'tool_calls']


## 9. `Tool` — Functions with a Schema

**What it is:** the wrapper that turns a Python function into something an LM can
discover and call — name, description, and a JSON argument schema derived from the signature
and docstring.

**How it works:** `dspy.Tool(func)` inspects the function; the adapter publishes the schema to
the model (as tools/functions); agent modules (ReAct, ReActV2, CodeAct) match the model's
choices back to the callable. You can override `name`/`desc`/`args` explicitly.

**When to use:** whenever an agent needs a capability — calculators, APIs, databases. This is
the same `Tool` type that powered the agents in Notebook 7.

In [9]:
import json


def get_exchange_rate(currency: str) -> float:
    """Get the USD exchange rate for a currency code like 'EUR'."""
    rates = {"EUR": 1.09, "JPY": 0.0066, "GBP": 1.27}
    return rates.get(currency.upper(), 1.0)


tool = dspy.Tool(get_exchange_rate)

print(f"Name: {tool.name}")
print(f"Description: {tool.desc}")
print("Argument schema the LM sees:")
print(json.dumps(tool.args, indent=2))
print(f"\nDirect call still works: get_exchange_rate('EUR') = {tool(currency='EUR')}")

Name: get_exchange_rate
Description: Get the USD exchange rate for a currency code like 'EUR'.
Argument schema the LM sees:
{
  "currency": {
    "type": "string"
  }
}

Direct call still works: get_exchange_rate('EUR') = 1.09


## 10. `ToolCalls` — The Record of What the Model Called

**What it is:** a structured record of one turn's tool invocations — each entry has
`name`, `args`, and an `id`. It's the type inside ReActV2's `tool_calls` output field and its
`History` events.

**How it works:** pydantic model over `tool_calls: list[ToolCall]`; validated on construction,
so malformed call records fail early instead of breaking the agent loop.

**When to use:** inspecting agent traces, building custom agent loops, evaluating tool-use
accuracy. **Pair with:** `Tool` (§9) for the callable side and `History` (§8) for the log.

In [10]:
calls = dspy.ToolCalls(
    tool_calls=[
        {"name": "get_exchange_rate", "args": {"currency": "EUR"}, "id": "call_0_0"},
        {"name": "get_exchange_rate", "args": {"currency": "JPY"}, "id": "call_0_1"},
    ]
)

for call in calls.tool_calls:
    print(f"{call.id}: {call.name}({call.args})")

print("\nThis is exactly what ReActV2's history.events contained in Notebook 7.")

call_0_0: get_exchange_rate({'currency': 'EUR'})
call_0_1: get_exchange_rate({'currency': 'JPY'})

This is exactly what ReActV2's history.events contained in Notebook 7.


## Part II — Tools

Where primitives are *data types*, tools are *utilities your program calls*:
retrievers that fetch knowledge, and a sandbox that runs code. Notebook 3 uses retrieval in a
full RAG pipeline; here you see the tools themselves, up close.

## 11. `Embeddings` — Embedding-Based Passage Retriever

**What it is:** retrieval over a text corpus using embedding similarity — `corpus` +
`embedder` in, top-`k` passages out.

**How it works:** embeds the corpus once at construction; queries are embedded per call and
scored by cosine similarity. Under 20k passages it brute-forces; above that it builds a FAISS
index for approximate candidates with exact re-ranking. Returns a `Prediction` with
`passages` and their corpus `indices`.

**Key params:** `corpus`, `embedder` (any `dspy.Embedder` — Notebook 8), `k`, `cache`,
`normalize`.

**When to use:** small-to-mid corpora, quick RAG prototypes, semantic search without a vector
DB. **Avoid when** you need filters, metadata, or >100k passages — use a real vector database.

In [11]:
# A real (tiny) corpus — the fact sheets behind the project's synthetic QA data
corpus = [
    "RAG combines retrieval with generation to ground answers in documents.",
    "Vector databases store embeddings for similarity search.",
    "DSPy is a framework for programming language models.",
    "Fine-tuning adapts a pre-trained model to a specific task.",
    "Chain-of-Thought prompting asks the model to show its reasoning.",
    "Attention mechanisms let models focus on relevant tokens.",
]

retriever = dspy.Embeddings(
    corpus=corpus,
    embedder=dspy.Embedder("openai/text-embedding-3-small"),
    k=3,
)

hits = retriever("what stores vectors for similarity search?")
print("Top-3 passages for 'what stores vectors for similarity search?':")
for rank, (passage, idx) in enumerate(zip(hits.passages, hits.indices, strict=True), start=1):
    print(f"  {rank}. [corpus #{idx}] {passage}")

Top-3 passages for 'what stores vectors for similarity search?':
  1. [corpus #1] Vector databases store embeddings for similarity search.
  2. [corpus #0] RAG combines retrieval with generation to ground answers in documents.
  3. [corpus #5] Attention mechanisms let models focus on relevant tokens.


## 12. `ColBERTv2` — Late-Interaction Retrieval

**What it is:** the retrieval tool DSPy was originally built around — queries a
**ColBERT server** (late-interaction neural retrieval) and returns Wikipedia-style passages.

**How it works:** `dspy.ColBERTv2(url=...)` points at a running ColBERT index server; each call
posts the query and gets back ranked passages with `long_text`. Preprocessing (lowercasing,
stripping articles) happens client-side. In 3.3.1 there is **no public default endpoint** —
you point it at your own server (or a team/college deployment).

**When to use:** strong-quality retrieval when you can host the index (classic choice for
academic/RAG demos). **Avoid when** you need managed uptime — prefer `Embeddings` or a vector
DB.

> ⚠️ **Environment note (verified):** the long-standing DSPy community demo server
> (`20.102.90.50:2017`) timed out from this machine at authoring time, so the live query is
> shown-not-run. The code below is the exact working pattern.

In [12]:
# 🚧 LIVE QUERY SHOWN FOR COMPLETENESS — the public demo server was unreachable
# from this environment when the notebook was authored (connection timeout).
retriever = dspy.ColBERTv2(url="http://20.102.90.50:2017/wiki17_abstracts")  # noqa: F841
# passages = retriever("When was the first successful human organ transplant?", k=3)
# for p in passages:
#     print(p["long_text"][:120])
#
# Self-hosting: dspy.ColBERTv2(url="http://localhost", port=2017) with a local index,
# e.g. from the ColBERT repo's indexing instructions.

## 13. `PythonInterpreter` — The Code Sandbox

**What it is:** DSPy's secure Python execution environment — the same WASM sandbox
(Deno + Pyodide) that powered `ProgramOfThought`, `CodeAct`, and `RLM` in Notebook 7.

**How it works:** code runs in an isolated interpreter with **no access to your filesystem,
network, or environment**; state persists across executions within a session; stdout is
captured back. Optional `tools` dict exposes host functions into the sandbox.

**When to use:** anytime you execute model-written code — always prefer this over `eval`/`exec`
on host Python. **Avoid when** you need heavy native libraries (numpy/pandas) — the WASM
subset is limited to the standard library.

In [13]:
from dspy.primitives import PythonInterpreter

with PythonInterpreter() as interp:
    # Real computation, isolated from the host process
    output = interp(
        "primes = [n for n in range(2, 50) if all(n % d for d in range(2, int(n**0.5) + 1))]"
        "\nprint(primes)"
    )
    print("Sandbox output:", output.strip())

    # State persists between executions in the same session
    interp.execute("total = sum(primes)")
    print("Follow-up sees prior state:", interp.execute("print(total)").strip())
print("\n(host filesystem and network were never reachable from that code)")

Sandbox output: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
Follow-up sees prior state: 328

(host filesystem and network were never reachable from that code)


## Cheat-Sheet

**Primitives**

| Type | Role | You've seen it in… |
|---|---|---|
| `Example` | labeled dataset atom | every trainset/devset (Notebooks 2, 9) |
| `Prediction` | universal module return | every `pred.answer` ever |
| `Image` | image input field | vision demos, Notebook in 02_outlines |
| `Audio` | audio input field | audio-capable models via `dspy.context(lm=...)` |
| `Code` | structured source-code field | code review/generation pipelines |
| `History` | typed conversation memory | ReActV2 traces (Notebook 7) |
| `Tool` | function + schema for agents | ReAct/ReActV2/CodeAct tools (Notebook 7) |
| `ToolCalls` | record of model-chosen calls | ReActV2 `tool_calls` field |

**Tools**

| Tool | What it does | Cost |
|---|---|---|
| `Embeddings` | top-k corpus passages by embedding similarity | embedding calls |
| `ColBERTv2` | top-k passages from a ColBERT server | free (self-hosted) |
| `PythonInterpreter` | sandboxed Python execution | free (local Deno) |

**Where it all connects:** primitives are the plumbing under Modules (Notebook 7) and Adapters
(Notebook 10); retrievers feed RAG (Notebook 3); the sandbox powers PoT/CodeAct/RLM
(Notebook 7).